# EXP-017 | Pseudo-labeling

test 예측 중 확신도 높은 샘플을 가짜 레이블로 train에 추가해 재학습.

```
Round 1: train(25만) → 모델 학습 → test 예측
         prob > 0.90 → pseudo label=1
         prob < 0.10 → pseudo label=0
Round 2: train + pseudo_test → 재학습 → 최종 예측
```

| 항목 | 내용 |
|---|---|
| 기반 모델 | LightGBM (EXP-004 파라미터, 속도 우선) |
| 기준선 | EXP-015 앙상블-v2 |
| CV | Stratified 5-Fold (원본 train에만 적용) |

In [9]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 17
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

# Pseudo-labeling 임계값
THRESHOLD_HIGH = 0.85  # 이 이상 → pseudo label = 1
THRESHOLD_LOW  = 0.10  # 이 이하 → pseudo label = 0

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [10]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. Round 1 — 원본 train으로 학습 → test 예측

In [11]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1,
    is_unbalance=True, learning_rate=0.05, num_leaves=127,
    min_child_samples=50, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=0.1,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_r1   = np.zeros(len(X_train))
test_r1  = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    oof_r1[val_idx] = model.predict(X_val)
    test_r1        += model.predict(X_test) / N_FOLDS

auc_r1 = roc_auc_score(y_train, oof_r1)
print(f'Round 1 OOF AUC: {auc_r1:.5f}')

Round 1 OOF AUC: 0.73864


## 3. Pseudo Label 선택

In [12]:
mask_pos  = test_r1 > THRESHOLD_HIGH
mask_neg  = test_r1 < THRESHOLD_LOW
mask      = mask_pos | mask_neg

pseudo_X = X_test[mask].copy()
pseudo_y = pd.Series((test_r1[mask] > 0.5).astype(int),
                      index=pseudo_X.index)

print(f'임계값: prob > {THRESHOLD_HIGH} → label=1 / prob < {THRESHOLD_LOW} → label=0')
print(f'Pseudo-labeled 샘플: {mask.sum():,}개  '
      f'(pos={mask_pos.sum():,}  neg={mask_neg.sum():,})')
print(f'전체 test 대비: {mask.mean()*100:.1f}%')
print(f'\n보강 후 train 크기: {len(X_train):,} → {len(X_train) + len(pseudo_X):,}')

임계값: prob > 0.85 → label=1 / prob < 0.1 → label=0
Pseudo-labeled 샘플: 13,784개  (pos=0  neg=13,784)
전체 test 대비: 15.3%

보강 후 train 크기: 256,351 → 270,135


## 4. Round 2 — 보강된 train으로 재학습

Pseudo-labeled 샘플은 **학습에만** 사용하고, OOF 평가는 원본 train 인덱스로만 계산.

In [13]:
# 보강 데이터셋 (pseudo 샘플은 항상 학습 fold에만 포함)
X_aug = pd.concat([X_train, pseudo_X], ignore_index=True)
y_aug = pd.concat([y_train.reset_index(drop=True), pseudo_y.reset_index(drop=True)],
                   ignore_index=True)

n_orig  = len(X_train)
oof_r2  = np.zeros(n_orig)   # 원본 train 인덱스만
test_r2 = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    # 원본 train의 fold 분할 기준 사용
    # 학습: 원본 train fold + 전체 pseudo 샘플
    pseudo_idx = np.arange(n_orig, len(X_aug))
    tr_idx_aug = np.concatenate([tr_idx, pseudo_idx])

    X_tr  = X_aug.iloc[tr_idx_aug]
    y_tr  = y_aug.iloc[tr_idx_aug]
    X_val = X_train.iloc[val_idx]   # 검증은 원본만
    y_val = y_train.iloc[val_idx]

    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    oof_r2[val_idx] = model.predict(X_val)
    test_r2        += model.predict(X_test) / N_FOLDS
    print(f'  Fold {fold}  train_size={len(X_tr):,}  '
          f'AUC={roc_auc_score(y_val, oof_r2[val_idx]):.5f}')

auc_r2   = roc_auc_score(y_train, oof_r2)
prauc_r2 = average_precision_score(y_train, oof_r2)
f1_r2    = f1_score(y_train, (oof_r2 >= 0.5).astype(int))
print(f'\nRound 2 OOF ROC-AUC: {auc_r2:.5f}')
print(f'Round 2 OOF PR-AUC : {prauc_r2:.5f}')
print(f'Round 2 OOF F1     : {f1_r2:.5f}')
print(f'Round 1 대비       : {auc_r2 - auc_r1:+.5f}')

  Fold 1  train_size=218,864  AUC=0.73686
  Fold 2  train_size=218,865  AUC=0.74153
  Fold 3  train_size=218,865  AUC=0.73891
  Fold 4  train_size=218,865  AUC=0.73764
  Fold 5  train_size=218,865  AUC=0.73973

Round 2 OOF ROC-AUC: 0.73885
Round 2 OOF PR-AUC : 0.44836
Round 2 OOF F1     : 0.51642
Round 1 대비       : +0.00021


## 5. 임계값 민감도 분석 (선택)

임계값에 따라 pseudo 샘플 수와 성능이 어떻게 달라지는지 확인.

In [14]:
print(f'현재 임계값 {THRESHOLD_HIGH}/{THRESHOLD_LOW} 결과: OOF AUC={auc_r2:.5f}')
print()
print('임계값별 pseudo 샘플 수 미리보기:')
print(f'{"임계값":>8}  {"pos":>8}  {"neg":>8}  {"합계":>8}  {"비율":>6}')
for th in [0.95, 0.90, 0.85, 0.80]:
    p = (test_r1 > th).sum()
    n = (test_r1 < (1-th)).sum()
    t = p + n
    print(f'{th:>8.2f}  {p:>8,}  {n:>8,}  {t:>8,}  {t/len(test_r1)*100:>5.1f}%')

현재 임계값 0.85/0.1 결과: OOF AUC=0.73885

임계값별 pseudo 샘플 수 미리보기:
     임계값       pos       neg        합계      비율
    0.95         0    13,054    13,054   14.5%
    0.90         0    13,784    13,784   15.3%
    0.85         0    15,038    15,038   16.7%
    0.80       192    16,712    16,904   18.8%


## 6. Submission 저장 & 실험 기록

In [15]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = test_r2
auc_str   = f'{auc_r2:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp017_조여진_07389.csv


In [16]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (oof_r2 >= 0.5).astype(int)
n_pseudo   = mask.sum()
params_str = (f'LGB + pseudo_label(th={THRESHOLD_HIGH}/{THRESHOLD_LOW}) | '
              f'pseudo={n_pseudo:,}샘플 | Round1={auc_r1:.5f} → Round2={auc_r2:.5f}')
NOTES    = f'Pseudo-labeling 2-Round. 임계값 {THRESHOLD_HIGH}/{THRESHOLD_LOW}, pseudo {n_pseudo:,}샘플 추가'
INSIGHTS = f'Round1→2: {auc_r2 - auc_r1:+.5f} | pos={mask_pos.sum():,} neg={mask_neg.sum():,}'

log_to_leaderboard(
    EXP_NO, AUTHOR, 'LGB+PseudoLabel', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    auc_r2, CV_STR, 'v1', X_train.shape[1], 'is_unbalance=True',
    'N', None, 'notebooks/13_pseudo_label_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-017 기록 완료 (row 15)
